# VWAP Statistical Model Research

## Phase 2 — Statistical Research Framework

### Research objectives

- Establish a reproducible historical research environment.
- Validate historical dataset quality and provenance.
- Reconstruct causal VWAP and probability-band observations.
- Reproduce the existing solid RMS and Gaussian ghost baselines.
- Evaluate statistical calibration and predictive performance.
- Establish chronological validation and out-of-sample protocols.

### Research principles

The existing engine is treated as a reference baseline rather
than an assumed optimal model.

Historical datasets must remain unchanged. Timestamp
conventions, overlapping observations, missing candles and
mixed-frequency data must be investigated before modelling.

Candidate distributions and trading rules must be evaluated
without look-ahead bias or retrospective parameter selection.

## 1. Research environment and dataset inventory

In [1]:

from pathlib import Path
import sys

import pandas as pd
import numpy as np


# Locate the repository root whether the notebook starts
# from the project directory or the notebooks directory.
PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "research_config.py").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.research_config import RESEARCH_CONFIG


DATA_DIR = RESEARCH_CONFIG["data_dir"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Python interpreter: {sys.executable}")
print(f"Instrument: {RESEARCH_CONFIG['instrument']}")
print(f"Timeframe: {RESEARCH_CONFIG['timeframe']}")
print(f"Baseline: {RESEARCH_CONFIG['baseline_dataset']}")


Project root: c:\GitHub Projects\VWAP-probability-band-engine-adaptive
Data directory: C:\GitHub Projects\VWAP-probability-band-engine-adaptive\data\historical
Python interpreter: c:\GitHub Projects\VWAP-probability-band-engine-adaptive\.venv\Scripts\python.exe
Instrument: US100.cash
Timeframe: M1
Baseline: 2025_full_year.csv


In [2]:

dataset_paths = sorted(DATA_DIR.glob("*.csv"))

assert dataset_paths, "No historical CSV datasets found."

dataset_inventory = pd.DataFrame(
    [
        {
            "filename": path.name,
            "size_mb": round(path.stat().st_size / 1024**2, 2),
            "columns": tuple(pd.read_csv(path, nrows=0).columns),
            "is_baseline": (
                path.name == RESEARCH_CONFIG["baseline_dataset"]
            ),
        }
        for path in dataset_paths
    ]
)

assert dataset_inventory["is_baseline"].sum() == 1, (
    "Expected exactly one baseline dataset."
)

print(f"Historical datasets: {len(dataset_inventory)}")
print(f"Total size: {dataset_inventory['size_mb'].sum():.2f} MB")

display(dataset_inventory)


Historical datasets: 9
Total size: 50.38 MB


,filename,size_mb,columns,is_baseline
0,2025_full_year.csv,28.41,"(datetime, open, high, low, close, tick_volume...",True
1,us100_cash_M1_new_york_2023_01_01_to_2024_01_0...,3.81,"(datetime, open, high, low, close, tick_volume...",False
2,us100_cash_M1_new_york_2024_01_01_to_2025_01_0...,3.98,"(datetime, open, high, low, close, tick_volume...",False
3,US100_cash_M1_NY_session_1y.csv,3.29,"(datetime, open, high, low, close, tick_volume...",False
4,US100_cash_M1_NY_session_30d.csv,0.43,"(datetime, open, high, low, close, tick_volume...",False
5,US100_cash_M1_NY_session_calm_2021_partial.csv,0.97,"(datetime, open, high, low, close, tick_volume...",False
6,US100_cash_M1_NY_session_recent_2025_2026.csv,3.29,"(datetime, open, high, low, close, tick_volume...",False
7,US100_cash_M1_NY_session_ukraine_war_2022_2023...,3.10,"(datetime, open, high, low, close, tick_volume...",False
8,US100_cash_M1_NY_session_volatile_2022.csv,3.10,"(datetime, open, high, low, close, tick_volume...",False
